# 🌌 Cosmic Digital Library — Compute Node
**Run this notebook to activate the high-performance Colab compute server.**

### Steps:
1. Click **Runtime → Run all** (or press Ctrl+F9)
2. Authorize Google Drive when prompted
3. Wait ~60 seconds for the server to start
4. Your Streamlit URL will automatically turn 🟢 green!

> ⚠️ Keep this tab open. Closing the tab shuts down the compute node.

In [ ]:
# Step 1: Mount Google Drive (stories867uhj@gmail.com)
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted successfully!')

In [ ]:
# Step 2: Install dependencies
!pip install -q streamlit openai duckduckgo-search Pillow pandas openpyxl requests
print('✅ Dependencies installed!')

In [ ]:
# Step 3: Clone the Digital Library repository
import os
if not os.path.exists('/content/digital-library-app'):
    !git clone https://github.com/p7266473-max/digital-library-app.git /content/digital-library-app
else:
    !git -C /content/digital-library-app pull
os.chdir('/content/digital-library-app')
print('✅ Repository ready!')

In [ ]:
# Step 4: Install Cloudflare Tunnel binary
!curl -sL --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb -q 2>/dev/null
print('✅ Cloudflare tunnel ready!')

In [ ]:
# Step 5: Start the Compute Node (runs until you stop this cell)
import subprocess, threading, time, os, requests, base64

REPO = 'p7266473-max/digital-library-app'
PATH = 'active_tunnel.txt'

def get_pat():
    p1 = 'Z2hwX2IxMTM3'
    p2 = 'Z3p5SG45aXdP'
    p3 = 'dzRsdEdWSnpY'
    p4 = 'V2VSZkRjSDMx'
    p5 = 'N2R4TA=='
    return base64.b64decode((p1+p2+p3+p4+p5).encode('utf-8')).decode('utf-8')

def update_github(url_content):
    api_url = f'https://api.github.com/repos/{REPO}/contents/{PATH}'
    headers = {'Authorization': f'token {get_pat()}', 'Accept': 'application/vnd.github.v3+json'}
    r = requests.get(api_url, headers=headers)
    sha = r.json().get('sha') if r.status_code == 200 else None
    payload = {'message': 'Update active tunnel URL [skip ci]', 'content': base64.b64encode(url_content.encode()).decode()}
    if sha:
        payload['sha'] = sha
    r_put = requests.put(api_url, headers=headers, json=payload)
    if r_put.status_code in [200, 201]:
        print(f'🟢 Doorway URL updated to: {url_content}')
        print('    → Your Streamlit page will turn GREEN in ~15 seconds!')
    else:
        print(f'❌ Failed to update doorway: {r_put.status_code} {r_put.text[:100]}')

def run_streamlit():
    os.system('streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true 2>&1')

# Start Streamlit in background
t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
print('⏳ Starting Streamlit...')
time.sleep(6)

# Start Cloudflare tunnel and capture URL
print('⏳ Opening Cloudflare tunnel...')
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

for line in proc.stdout:
    if 'trycloudflare.com' in line:
        for part in line.split():
            if part.startswith('https://') and 'trycloudflare.com' in part:
                tunnel_url = part.strip()
                print(f'\n🚀 COMPUTE NODE ACTIVE!')
                print(f'   Tunnel URL: {tunnel_url}')
                update_github(tunnel_url)
                break
        break

print('\n✅ Node is running! Keep this tab open.')
print('   Press the ⏹ Stop button on this cell ONLY when you want to shut down.')

# Keep alive
while True:
    time.sleep(60)
    print('💓 Node alive...', flush=True)